# Day 2 · Hands-on 1 — t-tests and ANOVA are linear models
**About 30 min · work in pairs** (one drives, one reads, predicts and questions — swap halfway)


Yesterday's tests were separate tools. Today we see that most of them are **one model**: outcome = intercept + effects + noise. Same numbers, one way of thinking.



> **Two AI roles — keep them apart**
> - 🧮 **Gemini in Colab = calculator.** Ask it to change, fix or explain *code*.
> - 🎓 **Claude / ChatGPT (other tab) = tutor you must check.** Ask it to *interpret*, then score its answer: **correct · complete · specific · actionable**.


**How this notebook works:** run cells from top to bottom. Each code cell has one line marked `# 🔧 CHANGE THIS` — the only thing you need to edit. Lines starting with 📋 go in the shared sheet. Lines starting with 🎯 appear only for the simulated dataset, where we know the true answer.

In [ ]:
# @title ▶️ Setup: run this cell first (no changes needed) { display-mode: "form" }
# Every analysis below uses ROLES (outcome, group, ...), never hard-coded column names.
# To add a dataset: write a loader function and add one entry to DATASETS.
%pip install -q lifelines
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats
from lifelines.datasets import load_gbsg2, load_lung

def _gbsg2():
    d = load_gbsg2()
    for c in ["tgrade", "menostat", "horTh"]:
        d[c] = d[c].astype(str)
    d["nodes4"] = (d["pnodes"] >= 4).astype(int)
    return d

def _lung():
    d = load_lung()
    d.columns = [c.replace(".", "_") for c in d.columns]
    d = d.dropna(subset=["ph_ecog"])
    d["ecog"] = d["ph_ecog"].map({0: "0", 1: "1", 2: "2+", 3: "2+"})
    d["sex"] = d["sex"].map({1: "male", 2: "female"})
    d["lost_weight"] = np.where(d["wt_loss"].isna(), np.nan, (d["wt_loss"] > 0).astype(float))
    return d

def _simulated(seed=2026, n=300):
    rng = np.random.default_rng(seed)
    genotype = rng.choice(["AA", "AB", "BB"], size=n, p=[0.4, 0.4, 0.2])
    sex = rng.choice(["F", "M"], size=n)
    age = rng.uniform(30, 80, n) + np.where(genotype == "BB", 8, 0)          # BB carriers are older
    g_eff = pd.Series({"AA": 0, "AB": 3, "BB": 8})[genotype].to_numpy()
    slope = np.where(sex == "M", 0.7, 0.4)                                    # interaction: steeper in men
    biomarker = 50 + slope * (age - 50) + g_eff + rng.normal(0, 6, n)
    logit = -1 + 0.05 * (age - 50) + pd.Series({"AA": 0, "AB": 0.4, "BB": 1.0})[genotype].to_numpy()
    disease = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    mu = np.exp(0.5 + 0.02 * (age - 50) + pd.Series({"AA": 0, "AB": 0.3, "BB": 0.7})[genotype].to_numpy())
    alpha = 0.8
    lesions = rng.negative_binomial(1 / alpha, 1 / (1 + alpha * mu))
    return pd.DataFrame(dict(biomarker=biomarker.round(1), genotype=genotype, sex=sex, age=age.round(1),
                             disease=disease, lesions=lesions))

def _my_data():
    # Upload a CSV to Colab (folder icon on the left) and edit the roles in DATASETS["my_data"]
    return pd.read_csv("my_data.csv")

DATASETS = {
    "gbsg2": dict(load=_gbsg2,
        about="GBSG2 breast cancer trial: 686 patients",
        outcome="tsize", outcome_label="Tumour size (mm)",
        group="tgrade", group_label="Tumour grade",
        x="pnodes", x_label="Positive lymph nodes",
        group2="menostat", group2_label="Menopausal status",
        binary="nodes4", binary_label="4 or more positive lymph nodes",
        binary_x="tsize", binary_group="tgrade",
        count="pnodes", count_label="Positive lymph nodes", count_x="tsize", count_group="tgrade"),
    "lung": dict(load=_lung,
        about="NCCTG advanced lung cancer study: 227 patients",
        outcome="pat_karno", outcome_label="Patient-rated Karnofsky score",
        group="ecog", group_label="ECOG performance status",
        x="age", x_label="Age (years)",
        group2="sex", group2_label="Sex",
        binary="lost_weight", binary_label="Lost weight in last 6 months",
        binary_x="age", binary_group="ecog",
        count=None),
    "simulated": dict(load=_simulated,
        about="Simulated biomarker study: 300 people, true effects known",
        outcome="biomarker", outcome_label="Biomarker level",
        group="genotype", group_label="Genotype",
        x="age", x_label="Age (years)",
        group2="sex", group2_label="Sex",
        binary="disease", binary_label="Disease present",
        binary_x="age", binary_group="genotype",
        count="lesions", count_label="Number of lesions", count_x="age", count_group="genotype",
        truth={
            "group": "AB vs AA = +3, BB vs AA = +8 (adjusted for age). BB carriers are also ~8 years older.",
            "x": "age slope = 0.4 per year in women, 0.7 in men",
            "interaction": "difference in age slopes (M vs F) = +0.3 per year",
            "binary": "odds ratio per year of age = 1.05; AB vs AA = 1.49; BB vs AA = 2.72",
            "count": "rate ratio per year of age = 1.02; AB vs AA = 1.35; BB vs AA = 2.01; overdispersed (alpha = 0.8)",
        }),
    "my_data": dict(load=_my_data,
        about="Your own CSV: edit these roles to match your column names",
        outcome="my_outcome", outcome_label="My outcome",
        group="my_group", group_label="My group",
        x="my_numeric_predictor", x_label="My numeric predictor",
        group2="my_two_level_group", group2_label="My two-level group",
        binary="my_yes_no_outcome", binary_label="My yes/no outcome (coded 0/1)",
        binary_x="my_numeric_predictor", binary_group="my_group",
        count=None),
}

def load_dataset(name):
    R = dict(DATASETS[name])
    d = R.pop("load")()
    for role in ["group", "group2", "binary_group", "count_group"]:
        if R.get(role):
            d[R[role]] = d[R[role]].astype(str)
    R["levels"] = sorted(d[R["group"]].dropna().unique())
    R.setdefault("truth", {})
    print(f"✅ {R['about']}  ({len(d)} rows)")
    print(f"   outcome: {R['outcome']} ({R['outcome_label']})   group: {R['group']} {R['levels']}")
    print(f"   numeric predictor: {R['x']}   second group: {R['group2']}")
    return d, R

def cat(var, ref):
    """Categorical term with an explicit reference level."""
    return f"C({var}, Treatment(reference='{ref}'))"

def nice(term):
    """Turn statsmodels term names into readable labels."""
    m = re.fullmatch(r"C\((\w+), Treatment\(reference='([^']+)'\)\)\[T\.([^\]]+)\](.*)", term)
    if m:
        var, ref, lvl, rest = m.groups()
        label = f"{var}: {lvl} vs {ref}"
        return label + (f" × {rest.lstrip(':')}" if rest else "")
    m = re.fullmatch(r"(\w+):C\((\w+), Treatment\(reference='([^']+)'\)\)\[T\.([^\]]+)\]", term)
    if m:
        x, var, ref, lvl = m.groups()
        return f"{x} × {var}: {lvl} vs {ref}"
    return term

def tidy(fit, exponentiate=False):
    """Coefficient table: estimate, 95% CI, p-value."""
    ci = fit.conf_int()
    t = pd.DataFrame({"estimate": fit.params, "CI low": ci[0], "CI high": ci[1], "p-value": fit.pvalues})
    t = t.drop(index=[i for i in t.index if i == "alpha"], errors="ignore")
    if exponentiate:
        t[["estimate", "CI low", "CI high"]] = np.exp(t[["estimate", "CI low", "CI high"]])
    t.index = [nice(i) for i in t.index]
    return t.round({"estimate": 3, "CI low": 3, "CI high": 3}).assign(**{"p-value": t["p-value"].map(lambda p: f"{p:.2g}")})

def show_truth(R, key):
    if key in R["truth"]:
        print(f"🎯 Truth used to simulate: {R['truth'][key]}")

print("✅ Setup complete")

## Choose a dataset
Everything in this notebook is written in terms of **roles** (an outcome, a group, a numeric predictor), so the same cells work on any dataset. Start with the one your facilitator names, then try `simulated`: it was generated with known effects, so you can check whether the model recovers the truth.

In [ ]:
# 🔧 CHANGE THIS: which dataset to use (all cells below adapt automatically)
dataset = "gbsg2"  # @param ["gbsg2", "lung", "simulated", "my_data"]

df, R = load_dataset(dataset)
df.head()

## A. A t-test is a linear model with one dummy variable
### 🧭 Concept
Code group membership as 0 (reference) or 1 (other). Fit outcome = b₀ + b₁ × dummy. Then **b₀ is the reference group's mean** and **b₁ is the difference in means**.

✍️ **Predict first:** will the t-test's p-value and the regression slope's p-value be similar, or identical?

In [ ]:
# 🔧 CHANGE THIS: the two groups to compare (any two levels printed above)
two_levels = R["levels"][:2]

y, g = R["outcome"], R["group"]
d = df[df[g].isin(two_levels)].dropna(subset=[y, g])
ref, other = two_levels
a, b = d.loc[d[g] == ref, y], d.loc[d[g] == other, y]

tt = stats.ttest_ind(b, a, equal_var=True)            # classic Student t-test
fit = smf.ols(f"{y} ~ {cat(g, ref)}", data=d).fit()   # linear model with a dummy for 'other'

print(f"Mean {y}:  {ref} = {a.mean():.2f}   {other} = {b.mean():.2f}   difference = {b.mean() - a.mean():.2f}\n")
print(f"Student t-test:  t = {tt.statistic:.3f}   p = {tt.pvalue:.4g}")
print(f"Linear model:    t = {fit.tvalues.iloc[1]:.3f}   p = {fit.pvalues.iloc[1]:.4g}\n")
display(tidy(fit))

plt.figure(figsize=(5, 3.5))
for i, (lv, vals) in enumerate([(ref, a), (other, b)]):
    plt.scatter(np.full(len(vals), i) + np.random.uniform(-0.15, 0.15, len(vals)), vals, alpha=0.3, s=12)
    plt.hlines(vals.mean(), i - 0.3, i + 0.3, color="black", linewidth=3)
plt.xticks([0, 1], [f"{ref}\n(intercept)", f"{other}\n(intercept + slope)"])
plt.ylabel(R["outcome_label"]); plt.show()

### ✅ Concept check
1. In the table, which number equals the mean of the reference group? Which equals the difference between groups?
2. The linear model matches the **Student** t-test (equal variances), not Welch's. Why might that be?

<details><summary>Show answers</summary>

1. `Intercept` = mean of the reference group; the second row = difference in means (other − reference).
2. The ordinary linear model assumes the same residual variance in every group, which is exactly the Student t-test's assumption. Welch drops it. (Model diagnostics in Hands-on 3 is how you check it.)
</details>

## B. One-way ANOVA is the same model with more dummies
### 🧭 Concept
With k groups the model gets k − 1 dummies, each comparing one group with the **reference**. The ANOVA F-test asks: are *all* those differences zero? Change the reference group: which numbers change, and which stay the same?

In [ ]:
# 🔧 CHANGE THIS: reference group (the baseline the other groups are compared with)
reference = R["levels"][0]

y, g = R["outcome"], R["group"]
d = df.dropna(subset=[y, g])
F, p = stats.f_oneway(*[d.loc[d[g] == lv, y] for lv in R["levels"]])
fit = smf.ols(f"{y} ~ {cat(g, reference)}", data=d).fit()

print(f"One-way ANOVA:        F = {F:.3f}   p = {p:.4g}")
print(f"Linear model F-test:  F = {fit.fvalue:.3f}   p = {fit.f_pvalue:.4g}\n")
print("Group means:", d.groupby(g)[y].mean().round(2).to_dict())
display(tidy(fit))
show_truth(R, "group")

### ✅ Concept check
When you changed the reference group, the coefficients changed but the F-test did not. Why?

<details><summary>Show answer</summary>

The model's fitted values (the group means) are identical whichever group is the baseline. Only the way we *describe* the differences changes. The F-test compares the whole model with "all groups equal", which doesn't depend on that choice.
</details>

**Note for the `simulated` dataset:** the 🎯 truth refers to effects *adjusted for age*. The unadjusted BB difference is larger, because BB carriers are older. That's the topic of the next hands-on.

## C. ⭐ If you have time: a correlation is a linear model too
### 🧭 Concept
Replace the groups with a numeric predictor. The slope's p-value equals the p-value of Pearson's correlation test.

In [ ]:
# 🔧 CHANGE THIS: numeric predictor (any numeric column)
predictor = R["x"]

y = R["outcome"]
d = df.dropna(subset=[y, predictor])
r, p_r = stats.pearsonr(d[predictor], d[y])
fit = smf.ols(f"{y} ~ {predictor}", data=d).fit()
print(f"Pearson correlation: r = {r:.3f}   p = {p_r:.4g}")
print(f"Linear model slope:  {fit.params[predictor]:.3f} per unit of {predictor}   p = {fit.pvalues[predictor]:.4g}")

plt.figure(figsize=(5, 3.5))
plt.scatter(d[predictor], d[y], alpha=0.3, s=12)
xs = np.linspace(d[predictor].min(), d[predictor].max(), 50)
plt.plot(xs, fit.params["Intercept"] + fit.params[predictor] * xs, color="black")
plt.xlabel(predictor); plt.ylabel(R["outcome_label"]); plt.show()

## 🤖 AI prompts
**🎓 Claude / ChatGPT (tutor) — a known failure point.** Run the cell below and paste the prompt it prints. It contains your own model output.

In [ ]:
# Generates a prompt from YOUR model in section B. Nothing to change.
row = tidy(fit_b := smf.ols(f"{R['outcome']} ~ {cat(R['group'], R['levels'][0])}", data=df).fit()).iloc[-1]
print(f"""My linear model of {R['outcome_label']} on {R['group_label']} gives an intercept of {fit_b.params.iloc[0]:.2f} and a coefficient of {row['estimate']} for "{row.name}". What is the average {R['outcome_label']} in that group, and what does the coefficient mean?""")

Score the answer with the rubric. A common slip is reading the coefficient as the group's *mean*, instead of its *difference from the reference*. A good answer computes intercept + coefficient.

**🧮 Gemini in Colab (calculator):** select the cell in section A and ask:
> *Also run Welch's t-test and print its p-value next to the others.*

Is Welch's p-value close to the others? When would it differ a lot?

## 💬 Discuss with your partner
- If t-tests and ANOVA are linear models, what do we gain by thinking of them that way? (Hint: what if you also want to account for age?)
- Which reference group would you choose for a paper, and why?
- What assumptions did the linear model make that the Mann–Whitney test from Day 1 does not?